In [ ]:
# 1. Import Dependencies
import pandas as pd
import numpy as np
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
import matplotlib.pyplot as plt

In [ ]:
# 2. Data Loading and Preprocessing
df = pd.read_csv("extracted_columns.csv")
X = df[['PSS-1', 'PSS-2', 'PSS-3', 'PSS-4', 'PSS-5', 'PSS-6', 'PSS-7', 
        'PSS-8', 'PSS-9', 'PSS-10', 'PSS-11', 'PSS-12', 'PSS-13', 'PSS-14']]
df['PSS_total'] = X.sum(axis=1)
y = df['PSS_total']

In [ ]:
# Perform 80:20 train-test split with fixed random seed
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# 3. Random Forest Hyperparameter Bounds
rfr_param_bounds = {
    'n_estimators': [50, 200],
    'max_depth': [None, 5, 10, 20, 50],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 5]
}


In [ ]:
# PSO particle creation
def create_particle():
    return {
        'position': [
            np.random.uniform(rfr_param_bounds['n_estimators'][0], rfr_param_bounds['n_estimators'][1]),  # Continuous n_estimators
            np.random.randint(0, len(rfr_param_bounds['max_depth'])),  # Discrete max_depth index
            np.random.uniform(rfr_param_bounds['min_samples_split'][0], rfr_param_bounds['min_samples_split'][1]),  # Continuous min_samples_split
            np.random.uniform(rfr_param_bounds['min_samples_leaf'][0], rfr_param_bounds['min_samples_leaf'][1])  # Continuous min_samples_leaf
        ],
        'velocity': [0]*4,
        'best_position': None,
        'best_fitness': -np.inf
    }

In [ ]:
# Decode particle positions to hyperparameters
def decode_particle_position(position):
    return {
        'n_estimators': int(np.clip(position[0], rfr_param_bounds['n_estimators'][0], rfr_param_bounds['n_estimators'][1])),
        'max_depth': rfr_param_bounds['max_depth'][int(np.clip(position[1], 0, len(rfr_param_bounds['max_depth'])-1))],
        'min_samples_split': int(np.clip(position[2], rfr_param_bounds['min_samples_split'][0], rfr_param_bounds['min_samples_split'][1])),
        'min_samples_leaf': int(np.clip(position[3], rfr_param_bounds['min_samples_leaf'][0], rfr_param_bounds['min_samples_leaf'][1]))
    }

# Fitness function for PSO: k-fold CV with RMSE
def fitness_rfr_kfold_pso(individual, X_train, y_train, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    rmse_scores = []
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = make_pipeline(
            StandardScaler(),
            RandomForestRegressor(
                n_estimators=individual['n_estimators'],
                max_depth=individual['max_depth'],
                min_samples_split=individual['min_samples_split'],
                min_samples_leaf=individual['min_samples_leaf'],
                random_state=42
            )
        )
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val_fold)
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))
    return -np.mean(rmse_scores)

# PSO hyperparameter tuning procedure with multiple runs
def pso_rfr_kfold_optimized(X_train, y_train, generations=10, swarm_size=20, runs=30, w=0.7, c1=1.5, c2=1.5, k=5):
    all_results = []
    for run in range(runs):
        print(f"\n=== PSO Run {run+1}/{runs} ===")
        np.random.seed(42 + run)
        random.seed(42 + run)
        swarm = [create_particle() for _ in range(swarm_size)]
        global_best_position = None
        global_best_fitness = -np.inf

        # Initialize fitness for all particles
        for particle in swarm:
            params = decode_particle_position(particle['position'])
            fitness = fitness_rfr_kfold_pso(params, X_train, y_train, k=k)
            particle['best_position'] = particle['position'][:]
            particle['best_fitness'] = fitness
            if fitness > global_best_fitness:
                global_best_fitness = fitness
                global_best_position = particle['position'][:]

        # PSO iterations over generations
        for gen in range(generations):
            for particle in swarm:
                for i in range(4):
                    r1, r2 = np.random.rand(), np.random.rand()
                    cognitive = c1 * r1 * (particle['best_position'][i] - particle['position'][i])
                    social = c2 * r2 * (global_best_position[i] - particle['position'][i])
                    particle['velocity'][i] = w * particle['velocity'][i] + cognitive + social
                    particle['position'][i] += particle['velocity'][i]

                params = decode_particle_position(particle['position'])
                fitness = fitness_rfr_kfold_pso(params, X_train, y_train, k=k)
                if fitness > particle['best_fitness']:
                    particle['best_fitness'] = fitness
                    particle['best_position'] = particle['position'][:]
                if fitness > global_best_fitness:
                    global_best_fitness = fitness
                    global_best_position = particle['position'][:]

            best_params = decode_particle_position(global_best_position)
            print(f"Generation {gen+1}, Best RMSE: {-global_best_fitness:.4f}, Params: {best_params}")

        # Train final model with best parameters found on full training set for this run
        best_individual = decode_particle_position(global_best_position)
        final_model = make_pipeline(
            StandardScaler(),
            RandomForestRegressor(
                n_estimators=best_individual['n_estimators'],
                max_depth=best_individual['max_depth'],
                min_samples_split=best_individual['min_samples_split'],
                min_samples_leaf=best_individual['min_samples_leaf'],
                random_state=42
            )
        )
        final_model.fit(X_train, y_train)
        preds_train = final_model.predict(X_train)

        rmse_train = np.sqrt(mean_squared_error(y_train, preds_train))
        mae_train = mean_absolute_error(y_train, preds_train)
        r2_train = r2_score(y_train, preds_train)

        print(f"Run {run+1} Metrics -> RMSE: {rmse_train:.4f}, MAE: {mae_train:.4f}, R^2: {r2_train:.4f}")

        all_results.append({
            'Run': run+1,
            'n_estimators': best_individual['n_estimators'],
            'max_depth': best_individual['max_depth'],
            'min_samples_split': best_individual['min_samples_split'],
            'min_samples_leaf': best_individual['min_samples_leaf'],
            'RMSE': rmse_train,
            'MAE': mae_train,
            'R2': r2_train
        })

    rmse_vals = [res['RMSE'] for res in all_results]
    mae_vals = [res['MAE'] for res in all_results]
    r2_vals = [res['R2'] for res in all_results]

    print("\n=== Final 30-Run PSO Summary ===")
    print(f"Average RMSE: {np.mean(rmse_vals):.4f} ± {np.std(rmse_vals):.4f}")
    print(f"Average MAE: {np.mean(mae_vals):.4f} ± {np.std(mae_vals):.4f}")
    print(f"Average R^2: {np.mean(r2_vals):.4f} ± {np.std(r2_vals):.4f}")

    results_df = pd.DataFrame(all_results)
    results_df.to_csv("PSO_RFR_30_Run_Results.csv", index=False)

    # Plot distributions of RMSE, MAE, and R^2 for all runs
    plt.figure(figsize=(12,6))
    plt.subplot(1,3,1)
    plt.boxplot(rmse_vals)
    plt.title("RMSE Distribution")
    plt.ylabel("RMSE")

    plt.subplot(1,3,2)
    plt.boxplot(mae_vals)
    plt.title("MAE Distribution")
    plt.ylabel("MAE")

    plt.subplot(1,3,3)
    plt.boxplot(r2_vals)
    plt.title("R^2 Distribution")
    plt.ylabel("R^2")

    plt.suptitle("PSO-Optimized RFR Performance over Multiple Runs")
    plt.tight_layout()
    plt.show()

    return results_df


In [ ]:
# Run PSO hyperparameter tuning with multiple runs
results_pso = pso_rfr_kfold_optimized(X_train, y_train, generations=10, swarm_size=20, runs=30)



=== PSO Run 1/30 ===
Generation 1, Best RMSE: 2.0696, Params: {'n_estimators': 114, 'max_depth': 20, 'min_samples_split': 4, 'min_samples_leaf': 1}
Generation 2, Best RMSE: 2.0537, Params: {'n_estimators': 95, 'max_depth': 20, 'min_samples_split': 3, 'min_samples_leaf': 1}
Generation 3, Best RMSE: 2.0475, Params: {'n_estimators': 117, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}
Generation 4, Best RMSE: 2.0444, Params: {'n_estimators': 114, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}
Generation 5, Best RMSE: 2.0444, Params: {'n_estimators': 114, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}
Generation 6, Best RMSE: 2.0444, Params: {'n_estimators': 114, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}
Generation 7, Best RMSE: 2.0444, Params: {'n_estimators': 114, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}
Generation 8, Best RMSE: 2.0444, Params: {'n_estimators': 114, 'max_depth': 50, 'min_sampl

In [ ]:
# Select best run based on lowest RMSE
best_run_pso = results_pso.loc[results_pso['RMSE'].idxmin()]
best_params_pso = {
    'n_estimators': best_run_pso['n_estimators'],
    'max_depth': best_run_pso['max_depth'],
    'min_samples_split': best_run_pso['min_samples_split'],
    'min_samples_leaf': best_run_pso['min_samples_leaf']
}
print(f"\nBest hyperparameters from PSO: {best_params_pso}")

In [ ]:
# Convert max_depth parameter safely (None if NaN)
max_depth_val = best_params_pso['max_depth']
if max_depth_val is None or (isinstance(max_depth_val, float) and np.isnan(max_depth_val)):
    max_depth_val = None
else:
    max_depth_val = int(max_depth_val)


Best hyperparameters from PSO: {'n_estimators': np.float64(200.0), 'max_depth': np.float64(nan), 'min_samples_split': np.float64(2.0), 'min_samples_leaf': np.float64(1.0)}


In [ ]:
# Train final model on full training data with best hyperparameters
final_pso_model = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(
        n_estimators=int(best_params_pso['n_estimators']),
        max_depth=max_depth_val,
        min_samples_split=int(best_params_pso['min_samples_split']),
        min_samples_leaf=int(best_params_pso['min_samples_leaf']),
        random_state=42
    )
)
final_pso_model.fit(X_train, y_train)


In [ ]:
# Evaluate final model on held-out validation set
y_val_pred_pso = final_pso_model.predict(X_val)

def evaluate_and_plot(model_name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"{model_name} RMSE: {rmse:.4f}")
    print(f"{model_name} MAE: {mae:.4f}")
    print(f"{model_name} R^2: {r2:.4f}")

    plt.figure(figsize=(8,6))
    plt.scatter(y_true, y_pred, alpha=0.6)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title(f'{model_name}: Actual vs Predicted')
    plt.show()

# Evaluate and plot results for final PSO-optimized Random Forest Regressor
evaluate_and_plot('Final PSO-Optimized Random Forest', y_val, y_val_pred_pso)